# Material and Contact definition for an Arch Problem

## Block Model from Arch Template

In [ ]:
from compas_dem.models import BlockModel
from compas_dem.templates import ArchTemplate

template = ArchTemplate(rise=3, span=10, thickness=0.3, depth=0.5, n=30)

model = BlockModel.from_template(template)

# Computing contact and assigning supports
model.compute_contacts()

# Nodes with only one contact are supports => degree of graphnode == 1
for element in model.elements():
    if model.graph.degree(element.graphnode) == 1:
        element.is_support = True

## Material Definition

In each block, a different material can be assigned as follows

In [ ]:

from compas_dem.material import Stone

In [ ]:
# Accessing predefined material properties
limestone = Stone.from_predefined_material('LimeStone')

print(f"Limestone properties: Ecm={limestone.Ecm}, poisson={limestone.poisson}, density={limestone.density}")

# Creating a custom material
custom_material = Stone(fc = 30,Ecm=5000, poisson=0.25, density=2500)

print(f"Custom material properties: Ecm={custom_material.Ecm}, poisson={custom_material.poisson}, density={custom_material.density}")


custom_mat = Stone.from_predefined_material("Generic")
custom_mat.Ecm = 5000
custom_mat.poisson = 0.25
custom_mat.density = 2500

print(f"Custom material properties: Ecm={custom_mat.Ecm}, poisson={custom_mat.poisson}, density={custom_mat.density}")


## Contact Definition

In each block, a different material can be assigned as follows

In [ ]:

from compas_dem.interactions import ContactProperties
from compas_dem.interactions import JointModel
from compas_dem.interactions import MohrCoulomb

In [ ]:

# ===============================================
# Prepare contact properties for the interfaces
# ===============================================

# Create a contact model (Mohr-Coulomb) with specific properties
mohr_columb = MohrCoulomb(phi= 30, c=0)
print(f"Contact model properties: phi={mohr_columb.phi}, c={mohr_columb.c}, mu={mohr_columb.mu}")

# Create a joint model with specific stiffness values
joint_a = JointModel(kn=10e10, kt=10e7)


# ================================
# Create contact properties based on the contact model and the joint model
# ================================
contact_type_1 = ContactProperties(contact_model=mohr_columb, joint_model=joint_a)


for idx, contact in enumerate(model.graph.edges()):
    # print(model.graph.edge_attribute(contact, "contacts")[0].__dict__)
    model.graph.edge_attribute(contact, "contact_properties", contact_type_1)
    # # print(f"Interface index: {intf.index(contact)}, contact properties: {model.graph.edge_attribute(contact, 'contact_properties')}")
    if idx == 5:
        print(f"Setting interface {contact} properties to: phi={mohr_columb.phi}, c={mohr_columb.c}, mu={mohr_columb.mu}")
        print(f"Interface {contact} joint stiffness: kn={joint_a.kn:.2e}, kt={joint_a.kt:.2e}")


## Force Assignment

Post processing forces is done through the Contact class in compas-dem, given the forces at each node belonging to every contact face, the resultant piint and force vector will be calculated

In [ ]:
import random

fn = [[random.uniform(0, 1) for _ in range(4)] for i in range(model.graph.number_of_edges())]
# ft_1 = [[random.uniform(0, 1) for _ in range(4)] for i in range(model.graph.number_of_edges())]
# ft_2 = [[random.uniform(0, 1) for _ in range(4)] for i in range(model.graph.number_of_edges())]
ft_1 = [[0 for _ in range(4)] for i in range(model.graph.number_of_edges())]
ft_2 = [[0 for _ in range(4)] for i in range(model.graph.number_of_edges())]
contact = model.graph
for intf in model.graph.edges():
    i = contact.edge_index()[intf]  # Get the index of the interface
    contact_face = contact.edge_attribute(intf, "contacts")[0]
    # print(f"Fn shape: {len(fn[1])}, Ft_1 shape: {len(ft_1)}, Ft_2 shape: {len(ft_2)}")
    contact_face.forces = [
        {"c_np": max(fn[i][0], 0.0), "c_nn": min(fn[i][0], 0.0), "c_u": ft_1[i][0], "c_v": ft_2[i][0]},
        {"c_np": max(fn[i][1], 0.0), "c_nn": min(fn[i][1], 0.0), "c_u": ft_1[i][1], "c_v": ft_2[i][1]},
        {"c_np": max(fn[i][2], 0.0), "c_nn": min(fn[i][2], 0.0), "c_u": ft_1[i][2], "c_v": ft_2[i][2]},
        {"c_np": max(fn[i][3], 0.0), "c_nn": min(fn[i][3], 0.0), "c_u": ft_1[i][3], "c_v": ft_2[i][3]},
    ]
for contact in model.contacts():
    contact.resultantforce

### Reading the results and visualizing after saving into the force dictionary

[https://blockresearchgroup.github.io/compas_dem/latest/tutorial/dem_datastructure_results.html]

In [ ]:
from compas_viewer.viewer import Viewer

import compas.geometry as cg
from compas.colors import Color

viewer = Viewer()

# Add elements
for element in model.elements():
    viewer.scene.add(element.modelgeometry, show_faces=False)

for contact in model.contacts():
    viewer.scene.add(contact.polygon, facecolor=Color.cyan().lightened(50))

    # Contact force result is presented as a line, with the line's midpoint at the contact polygon's centroid.
    # The line's length proportional to the force magnitude.
    L = contact.resultantforce[0]
    
    # Scaling down the line for better visualization
    vec = cg.Vector(*L.midpoint)
    L.translate(-vec)
    L.scale(0.3, 0.3, 0.3)
    L.translate(vec)
    
    # Adding the line to the scene with a specific color and scale
    viewer.scene.add(L, color=Color.red(), scale=0.5)

viewer.show()

## Visualisation

In [ ]:
# from compas_notebook.viewer import Viewer

# viewer = Viewer()
# viewer.scene.add(model)
# viewer.show()
